# Neural Networks + PyTorch


## Libraries


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from random import random
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## GPU


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")


Device: mps


## Data


In [3]:
def generate_dataset(num_samples, test_size):
    x = np.array([[random() / 2 for _ in range(2)] for _ in range(num_samples)])
    y = np.array([[i[0] + i[1]] for i in x])
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_size)

    return x_train, x_test, y_train, y_test

In [4]:
x_train, x_test, y_train, y_test = generate_dataset(10, 0.2)
print(f"x_test: {x_test}")
print(f"y_test: {y_test}")

x_test: [[0.00299581 0.43522591]
 [0.47943567 0.3223021 ]]
y_test: [[0.43822172]
 [0.80173778]]


## PyTorch Multi-Layer Perceptron


In [5]:
# build model
class SumMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # build a model: 2 -> 5 -> 1
        self.net = nn.Sequential(
            nn.Linear(2, 5), nn.Sigmoid(), nn.Linear(5, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


# train model
def train(model, dataloader, optimizer, loss_fn, epochs=100):
    model.train()
    for epoch in range(epochs):
        # calculate the average loss per batch
        total_loss = 0
        for x_batch, y_batch in dataloader:
            optimizer.zero_grad()  # clear the gradientsts
            preds = model(x_batch)  # forward propagation
            loss = loss_fn(preds, y_batch)  # mse loss
            loss.backward()  # backward propagation
            optimizer.step()  # compute the gradient descent to update the parameters
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss / len(dataloader):.4f}"
            )
    print(f"---------------------------")
    print("Training Finished!\n")


# evaluate model
def evaluate(model, x_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(x_test)
        loss = nn.MSELoss()(
            preds, y_test
        )  # calculate the MSE loss from the entire test set
    print(f"Evaluation MSE Loss: {loss.item():.4f}\n")

## Testing

In [6]:
# data
x_train, x_test, y_train, y_test = generate_dataset(5000, 0.2)
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# dataloader
dataset = TensorDataset(x_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# model, optimizer, loss
model = SumMLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# train
train(model, dataloader, optimizer, loss_fn, epochs=100)

# evaluation
evaluate(model, x_test, y_test)

# predictions
data = torch.tensor([[0.1, 0.2], [0.2, 0.2]], dtype=torch.float32)
model.eval()
with torch.no_grad():
    predictions = model(data)

print(f"Predictions")
for d, p in zip(data, predictions):
    print(f"{d[0].item()} + {d[1].item()} = {p[0].item():.4f}")


Epoch 10/100 | Loss: 0.0409
Epoch 20/100 | Loss: 0.0394
Epoch 30/100 | Loss: 0.0363
Epoch 40/100 | Loss: 0.0305
Epoch 50/100 | Loss: 0.0219
Epoch 60/100 | Loss: 0.0129
Epoch 70/100 | Loss: 0.0065
Epoch 80/100 | Loss: 0.0031
Epoch 90/100 | Loss: 0.0015
Epoch 100/100 | Loss: 0.0009
---------------------------
Training Finished!

Evaluation MSE Loss: 0.0009

Predictions
0.10000000149011612 + 0.20000000298023224 = 0.3114
0.20000000298023224 + 0.20000000298023224 = 0.4031
